# Solar filament instance segmentation — end-to-end pipeline

This notebook reproduces our Kaggle submission for the *Solar Filament Segmentation Challenge 2026*
from the released weights (no training needed):

1. **input** — each GONG H-alpha JPEG (2048 x 2048, grayscale) is resized to the model resolution and
   standardised with robust statistics of the solar disk (median / IQR), off-disk pixels set to a constant;
2. **segmentation network** — U-Net(s) with an ImageNet-pretrained encoder predict the probability that a
   pixel belongs to a filament (trained on the mean of the annotators' masks); test-time augmentation (the 8
   flips / transpositions at 1536 px plus the 4 flips at native 2048 px) and averaging of the 5 cross-validation
   models of each of the two encoders (ResNet-34, EfficientNetV2-S);
3. **instances** — the probability is resized to 2048 x 2048, thresholded, split into 8-connected components
   and hole-filled; every component is a candidate filament (instances never overlap);
4. **instance re-scoring** — a gradient-boosting model, fit on out-of-fold candidates, estimates the chance that a
   candidate matches an annotated filament (IoU > 0.5) from its size, shape and probability statistics;
   candidates below the threshold are dropped;
5. **output** — one COCO-RLE row per filament (`filament_id`, `segmentation_rle`).

The default configuration `configs/final_all.json` reproduces our final submission. `configs/final.json`
(4 flips at 1536 px only, about 4x less computation, slightly lower score) can be selected with the environment variable
`FILAMENT_CONFIG`.

Section 6 re-computes the exact competition metric (Panoptic Quality) on training images to show how it is scored.

In [ ]:
# 0. setup (run once). Paths can be overridden with environment variables.
import os, sys, glob, time
REPO = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src'))
DATA = os.environ.get('FILAMENT_DATA', '/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026')
if not os.path.isdir(DATA):  # find the competition folder wherever it is mounted
    hits = glob.glob('/kaggle/input/**/MAGFiLO_1.0_Kaggle_2026', recursive=True) + glob.glob(os.path.join(REPO, 'data', '**', 'MAGFiLO_1.0_Kaggle_2026'), recursive=True)
    DATA = hits[0] if hits else DATA
TEST_DIR = os.path.join(DATA, 'test', 'test_images')
CONFIG = os.path.join(REPO, os.environ.get('FILAMENT_CONFIG', 'configs/final_all.json'))  # default = final submission; relative to the repo root or absolute
print('repo', REPO, '| data', DATA, '| config', CONFIG)

In [ ]:
# 1. load the ensemble described in the configuration (default configs/final_all.json)
import json, numpy as np, pandas as pd, torch
from filseg.pipeline import Pipeline
print(json.dumps(json.load(open(CONFIG)), indent=1))
pipe = Pipeline(CONFIG)
print('device', pipe.device, '| models per resolution', {k: len(v) for k, v in pipe.groups.items()})

In [ ]:
# 2-5. predict every test image and write the submission
names = sorted(glob.glob(os.path.join(TEST_DIR, '*.jpeg')))
names = names[:int(os.environ.get('MAX_IMAGES', len(names)))]  # quick smoke test: MAX_IMAGES=2
rows, t0 = [], time.time()
for i, path in enumerate(names):
    rows += pipe.rows(path)
    if (i + 1) % 30 == 0:
        print(f'{i + 1}/{len(names)} images, {time.time() - t0:.0f}s')
sub = pd.DataFrame(rows, columns=['filament_id', 'segmentation_rle'])
sub.to_csv('submission.csv', index=False)
print(len(sub), 'filaments in', len(names), 'images ->', os.path.abspath('submission.csv'))

In [ ]:
# sanity checks: unique ids, every mask decodes, no overlaps between the filaments of an image
from filseg.data import counts_to_mask
assert sub.filament_id.is_unique
for img, g in sub.groupby(sub.filament_id.str.split('_').str[0]):
    acc = np.zeros((2048, 2048), np.uint8)
    for c in g.segmentation_rle:
        m = counts_to_mask(c)
        assert m.sum() > 0 and not (acc & m).any()
        acc |= m
print('submission OK')

In [ ]:
# visual check on a few test images: probability map and final instances
import matplotlib.pyplot as plt, cv2
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax_col, path in zip(axes.T, names[:3]):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    prob = pipe.probability(path); lab, n = pipe.segment(prob)
    ax_col[0].imshow(img, cmap='gray'); ax_col[0].imshow(np.ma.masked_less(prob, 0.2), cmap='autumn', alpha=0.6, vmin=0, vmax=1)
    ax_col[0].set_title(os.path.basename(path) + ' probability'); ax_col[0].axis('off')
    ax_col[1].imshow(img, cmap='gray'); ax_col[1].imshow(np.ma.masked_equal(lab, 0) % 20, cmap='tab20', alpha=0.7, interpolation='nearest')
    ax_col[1].set_title(f'{n} filaments'); ax_col[1].axis('off')
plt.tight_layout(); plt.show()

## 6. Exact competition metric on training images

`filseg.metric` re-implements the organisers' self-evaluation notebook: every *annotator-image* of the ground truth
is scored separately against all predictions of that image, a pair is a true positive when IoU > 0.5, and
PQ = sum of TP IoUs / (TP + FP/2 + FN/2) pooled over all annotator-images (`tests/test_metric_matches_official.py`
checks it against a literal port of the official code). Training images were seen by the released models, so the
score below is optimistic; the honest numbers are the out-of-fold ones in the report.

In [ ]:
from filseg.data import load_coco, build_gt_table
from filseg.metric import GTCache, PQStats
coco = load_coco(os.path.join(DATA, 'train', 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
gt = pd.DataFrame(build_gt_table(coco))
some = sorted(gt.image.unique())[:int(os.environ.get('MAX_TRAIN_IMAGES', 20))]
cache = GTCache(gt[gt.image.isin(some)].to_dict('records'))
tot = PQStats()
for img in some:
    lab, n = pipe.segment(pipe.probability(os.path.join(DATA, 'train', 'train_images', img + '.jpeg')))
    for s in cache.score(img, lab, n).values():
        tot.add(s)
print(f'PQ {tot.pq:.4f}  SQ {tot.sq:.4f}  RQ {tot.rq:.4f}  TP {tot.tp} FP {tot.fp} FN {tot.fn}  ({len(some)} training images, in-sample)')